# MIST — Molecular Insight SMILES Transformer

A family of molecular foundation models for property prediction, pre-trained
with masked language modelling on [Smirk](https://github.com/BattModels/smirk)-tokenized
SMILES from Enamine REAL Space, then fine-tuned for downstream tasks.

- Code: https://github.com/BattModels/mist
- Paper: https://arxiv.org/abs/2510.18900
- Models: https://huggingface.co/mist-models

This notebook runs inside the `mist` kernel, which is the MIST container.
The working directory is `/opt/mist`. `hf_cache/` and `output/` are bound in
from project space; everything else in the container is read-only.

The workflows below are adapted from the upstream tutorials in
[BattModels/mist-demo](https://github.com/BattModels/mist-demo), with the
checkpoint loading changed to read from the local offline cache.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoModel
from rdkit import Chem
from rdkit.Chem import Draw
import warnings
warnings.filterwarnings("ignore")

device = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch:", torch.__version__)
print("Using device:", device)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

## Checkpoints are loaded offline

NeSI compute nodes have no outbound internet access, so the container sets
`HF_HUB_OFFLINE=1` and points `HF_HOME` at a bind-mounted cache that
`build_mist.sh` populated on a login node.

If a model you want is missing, add it to `MODELS` in `build_mist.sh` and
re-run that script from a login node. Calling `from_pretrained()` on an
uncached model here will fail rather than hang.

MIST-26.9M is small enough that CPU inference is viable, so these cells work
without a GPU — just more slowly.

In [ ]:
import os
print("HF_HOME:", os.environ.get("HF_HOME"))
print("HF_HUB_OFFLINE:", os.environ.get("HF_HUB_OFFLINE"))
!ls hf_cache/hub 2>/dev/null || echo "cache empty - run build_mist.sh on a login node"

## Molecular property prediction

Three fine-tuned MIST models predict boiling point, flash point and melting
point. We sweep chain length for linear 1-alkenes and 1-alcohols and check
the trends come out chemically sensible.

Upstream tutorial: [molecular_property_prediction.ipynb](https://github.com/BattModels/mist-demo/blob/main/tutorials/molecular_property_prediction.ipynb)

In [ ]:
models_info = {
    "Boiling Point": "mist-models/mist-26.9M-b302p09x-bp",
    "Flash Point":   "mist-models/mist-26.9M-cyuo2xb6-fp",
    "Melting Point": "mist-models/mist-26.9M-y3ge5pf9-mp",
}

# trust_remote_code=True executes custom modelling code shipped with the
# checkpoint. build_mist.sh downloaded that code into hf_cache/ already.
loaded_models = {}
for prop_name, model_path in models_info.items():
    print(f"Loading {prop_name} model...")
    loaded_models[prop_name] = (
        AutoModel.from_pretrained(model_path, trust_remote_code=True).to(device).eval()
    )

In [ ]:
def generate_alkene_smiles(n):
    """Linear 1-alkene with n carbons."""
    if n == 2:
        return "C=C"
    if n == 3:
        return "C=CC"
    return "C=C" + "C" * (n - 2)

def generate_alcohol_smiles(n):
    """Linear 1-alcohol with n carbons."""
    return "CO" if n == 1 else "C" * (n - 1) + "O"

chain_lengths = list(range(2, 25, 2))
alkenes  = {"carbons": chain_lengths, "smiles": [generate_alkene_smiles(n)  for n in chain_lengths]}
alcohols = {"carbons": chain_lengths, "smiles": [generate_alcohol_smiles(n) for n in chain_lengths]}
print("alkenes :", alkenes["smiles"][:4], "...")
print("alcohols:", alcohols["smiles"][:4], "...")

In [ ]:
def predict_all(series):
    out = {}
    for prop_name, model in loaded_models.items():
        predictions = model.predict(series["smiles"])
        prop_key = list(predictions.keys())[0]
        out[prop_name] = predictions[prop_key]["value"].cpu().numpy().tolist()
    return out

alkene_predictions  = predict_all(alkenes)
alcohol_predictions = predict_all(alcohols)
for p in models_info:
    print(f"{p}: alkene C2 = {alkene_predictions[p][0]:.1f}, alcohol C2 = {alcohol_predictions[p][0]:.1f}")

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(8, 10), sharex=True)

for ax, prop_name in zip(axes, models_info):
    ax.plot(alkenes["carbons"],  alkene_predictions[prop_name],
            "o-", color="#1f77b4", linewidth=2, markersize=7, label="Alkenes")
    ax.plot(alcohols["carbons"], alcohol_predictions[prop_name],
            "o-", color="#ff7f0e", linewidth=2, markersize=7, label="Alcohols")
    ax.set_title(prop_name)
    ax.set_ylabel("Temperature (\u00b0C)")
    ax.grid(True, alpha=0.3)

axes[0].legend()
axes[-1].set_xlabel("Carbon chain length")
plt.tight_layout()
plt.savefig("output/property_trends.png", dpi=150)
plt.show()

The model extrapolates chemically reasonable trends: all three properties rise
with chain length as molar mass and dispersion forces increase, and alcohols
sit above alkenes throughout, reflecting hydrogen bonding.

Note these are **predictions from a fine-tuned model**, not measurements —
treat the absolute values with appropriate caution, especially out at C24
where the fine-tuning data is sparse.

## Olfaction: predicting molecular scent

A multi-label model over scent descriptors. The classic demonstration is
acetone versus thioacetone — one atom apart, famously very different smells.

Upstream tutorial: [olfaction_prediction.ipynb](https://github.com/BattModels/mist-demo/blob/main/tutorials/olfaction_prediction.ipynb)

In [ ]:
# Helpers from the upstream mist-demo olfaction tutorial.
def kekulize_smiles(smiles):
    """The olfaction model was trained on kekulized SMILES."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f"Invalid SMILES: {smiles}")
    Chem.Kekulize(mol)
    return Chem.MolToSmiles(mol, kekuleSmiles=True)

def get_top_scents(predictions, threshold=0.5, top_n=None):
    """Sigmoid the per-descriptor logits and return the strongest."""
    scents = []
    for descriptor, pred_dict in predictions.items():
        logit = pred_dict["value"][0]
        if hasattr(logit, "item"):
            logit = logit.item()
        prob = 1 / (1 + np.exp(-logit))
        if prob >= threshold:
            scents.append((descriptor, float(prob)))
    scents.sort(key=lambda x: x[1], reverse=True)
    return scents[:top_n] if top_n else scents

In [ ]:
odour_model = AutoModel.from_pretrained(
    "mist-models/mist-26.9M-48kpooqf-odour", trust_remote_code=True
).to(device).eval()

In [ ]:
molecules = {
    "Acetone":      "CC(=O)C",
    "Thioacetone":  "CC(=S)C",
    "Vanillin":     "COc1cc(C=O)ccc1O",
    "Limonene":     "CC(=C)C1CCC(=CC1)C",
}

for name, smiles in molecules.items():
    predictions = odour_model.predict([kekulize_smiles(smiles)])
    print(f"\n{name}  ({smiles})")
    for descriptor, prob in get_top_scents(predictions, threshold=0.0, top_n=5):
        print(f"   {descriptor:<20} {prob:.3f}")

In [ ]:
# Draw the structures alongside the predictions
Draw.MolsToGridImage(
    [Chem.MolFromSmiles(s) for s in molecules.values()],
    legends=list(molecules.keys()),
    molsPerRow=4,
    subImgSize=(220, 180),
)

## Your own molecules

Replace the SMILES below with whatever you are working on. Every fine-tuned
MIST checkpoint exposes the same `model.predict(list_of_smiles)` interface,
so the same pattern covers any of them.

In [ ]:
my_smiles = [
    "CCO",                       # ethanol
    "c1ccccc1",                  # benzene
    "CC(=O)Oc1ccccc1C(=O)O",     # aspirin
]

import pandas as pd
rows = []
for smi in my_smiles:
    row = {"smiles": smi}
    for prop_name, model in loaded_models.items():
        predictions = model.predict([smi])
        prop_key = list(predictions.keys())[0]
        row[prop_name] = float(predictions[prop_key]["value"][0])
    rows.append(row)

df = pd.DataFrame(rows)
df.to_csv("output/my_predictions.csv", index=False)
df

In [ ]:
!ls -lh output/